In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Samplerオプションの指定

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="パッケージバージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
オプションを使用してSamplerプリミティブをカスタマイズできます。このセクションでは、Qiskit Runtimeプリミティブのオプションを指定する方法に焦点を当てています。プリミティブの`run()`メソッドのインターフェースはすべての実装で共通ですが、オプションは共通ではありません。[`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2)および[`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html)のオプションについては、対応するAPIリファレンスを参照してください。

<span id="pass-options"></span>
## Samplerオプションの設定
Samplerの初期化時、初期化後、またはSamplerの初期化後にオプションを更新することができます。これらの方法の使用方法については、[オプションの概要](/guides/runtime-options-overview#options-precedence)トピックを参照してください。

また、次のセクションで説明するように、`run()`メソッドで`shots`の値を設定することもできます。
<span id="run-method"></span>
### Run()メソッド
`run()`に渡せる値は、インターフェースで定義されたもの、つまり`shots`のみです。これにより、現在の実行に対して`default_shots`に設定されたすべての値が上書きされます。

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### 特殊なケース
<span id="shots"></span>
#### ショット数
`SamplerV2.run`メソッドは2つの引数を受け取ります。PUBのリスト（各PUBはPUB固有のショット数を指定できます）と、shotsキーワード引数です。これらのショット値はSampler実行インターフェースの一部であり、RuntimeSamplerのオプションとは独立しています。Samplerの抽象化に準拠するために、オプションとして指定された値よりも優先されます。

ただし、`shots`がPUBによっても`run`キーワード引数でも指定されていない場合（またはすべてが`None`の場合）、オプションのshots値が使用されます。特に`default_shots`が使用されます。

まとめると、Samplerにおける特定のPUBのshots指定の優先順位は次のとおりです。

1. PUBがshotsを指定している場合、その値を使用します。
2. `run`で`shots`キーワード引数が指定されている場合、その値を使用します。
4. `twirling`が有効な場合（デフォルトでTrue）、[`twirling`オプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)で指定された`num_randomizations`と`shots_per_randomization`の積が使用されます。
5. `sampler.options.default_shots`が指定されている場合、その値を使用します。

したがって、可能なすべての場所にshotsが指定されている場合、最高優先度のもの（PUBで指定されたshots）が使用されます。

例：